# Task 01 — full Colab validation

Use a **CPU runtime**. This notebook clones one published revision, installs its locked requirements, runs the complete test gate, and only then exposes the 30-seed experiment. `RUN_FULL` is false so **Run all is safe**. Nothing here authenticates to or pushes to GitHub.

In [ ]:
REPO_URL = "https://github.com/PaulsonLab/energy-inference-bo.git"
REPO_REF = "main"  # Prefer a published commit SHA for an archival run.
RUN_FULL = False      # Change to True only after the test cell passes.
PROFILE = "full"


In [ ]:
import platform, sys
version = sys.version_info[:2]
if version not in {(3, 11), (3, 12)}:
    raise RuntimeError(f"Python {platform.python_version()} is unsupported; use Colab Python 3.11 or 3.12.")
print("Python", platform.python_version())


In [ ]:
from pathlib import Path
import subprocess
REPO_DIR = Path("/content/energy-inference-bo")
if REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} already exists. Restart the runtime for a clean archival run.")
subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Checked out", GIT_SHA)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "pytest==9.1.1"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)


In [ ]:
import importlib.metadata as metadata
for package in ["energy-inference-bo", "torch", "botorch", "gpytorch", "numpy", "scipy"]:
    print(package, metadata.version(package))
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


## Explicit full run

The next cell runs 30 CPU seeds. Set `RUN_FULL = True` in the configuration cell only after tests pass. Outputs stay under ignored `artifacts/task01/full/`.

In [ ]:
if not RUN_FULL:
    raise RuntimeError("Full run is disabled. Review the configuration, set RUN_FULL=True, and rerun this cell.")
OUTPUT_DIR = REPO_DIR / "artifacts/task01/full"
COMMAND = [sys.executable, "-m", "energy_bo.experiments.run_task01", "--seeds", *[str(seed) for seed in range(30)], "--output-dir", str(OUTPUT_DIR), "--report-path", str(OUTPUT_DIR / "SUMMARY.md")]
subprocess.run(COMMAND, cwd=REPO_DIR, check=True)


In [ ]:
import json, shutil
manifest = {
    "task": "task01", "profile": PROFILE, "git_sha": GIT_SHA,
    "python": platform.python_version(), "accelerator": "cpu",
    "packages": {name: metadata.version(name) for name in ["torch", "botorch", "gpytorch", "numpy", "scipy"]},
    "command": COMMAND,
}
(OUTPUT_DIR / "colab_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
archive = Path(shutil.make_archive("/content/task01_full_outputs", "zip", root_dir=REPO_DIR, base_dir="artifacts/task01/full"))
print(archive, archive.stat().st_size, "bytes")
from google.colab import files
files.download(str(archive))


## What to do with the ZIP

Unzip it locally and keep the `full/` directory intact while reviewing. In your Git clone, create `results/task01/full/` and copy only `SUMMARY.md`, `task01_config.json`, `task01_metrics.csv`, `colab_manifest.json`, and the one or two decisive figures. Keep redundant generated files local. Commit and push manually; the notebook has made no GitHub changes.